# Retrain the road-defect detector (one click)

**What to do:** `Runtime → Change runtime type → T4 GPU → Save`, then `Runtime → Run all`.
Sign in to Google when asked (to read your datasets from Drive). It takes about 1–2 hours.

At the end, `defects.onnx` and `defects_metrics.json` are saved in your Drive folder `road_models/`.
Copy both into the app's `models/` folder (replace the old files) and redeploy.

In [ ]:
# 1. Settings — change only if needed
DATASETS_FOLDER = "/content/drive/MyDrive/road_datasets"  # folder that contains your 6 dataset folders
BASE_MODEL = "yolo11s.pt"   # or "yolo11m.pt" (more accurate, slower on the server)
EPOCHS = 150
IMGSZ = 640                 # the server runs at 640
REPO = "https://github.com/ERpdeore/indian-road-capacity-analyzer.git"  # EDIT after you create the repo

In [ ]:
# 2. Connect Google Drive and install
from google.colab import drive
drive.mount('/content/drive')
!pip -q install ultralytics onnx onnxslim
!git clone -q $REPO /content/app || echo "Repo not found: upload training/ files manually"
%cd /content/app/training

In [ ]:
# 3. Merge the datasets (classes matched by name, polygons -> boxes, no train/valid leakage)
import yaml
cfg = yaml.safe_load(open('class_map.yaml')); cfg['root'] = DATASETS_FOLDER
yaml.safe_dump(cfg, open('class_map_colab.yaml','w'), sort_keys=False)
!python merge_datasets.py --config class_map_colab.yaml --out /content/merged

In [ ]:
# 4. Train
from ultralytics import YOLO
model = YOLO(BASE_MODEL)
model.train(data='/content/merged/data.yaml', imgsz=IMGSZ, epochs=EPOCHS, patience=30, batch=-1,
            cos_lr=True, close_mosaic=15, degrees=0, fliplr=0.5, flipud=0, hsv_s=0.6, hsv_v=0.4,
            seed=0, name='defects', exist_ok=True)

In [ ]:
# 5. Evaluate on the validation split and export for the server
import json, shutil, os
best = YOLO('runs/detect/defects/weights/best.pt')
m = best.val(data='/content/merged/data.yaml', imgsz=IMGSZ)
metrics = {"precision": float(m.box.mp), "recall": float(m.box.mr), "mAP50": float(m.box.map50),
           "mAP50_95": float(m.box.map), "base": BASE_MODEL, "epochs": EPOCHS, "imgsz": IMGSZ,
           "per_class_mAP50": {best.names[i]: float(v) for i, v in enumerate(m.box.ap50)},
           "classes": [best.names[i] for i in sorted(best.names)]}
print(json.dumps(metrics, indent=1))
onnx = best.export(format='onnx', imgsz=IMGSZ, opset=12, simplify=True)
os.makedirs('/content/drive/MyDrive/road_models', exist_ok=True)
shutil.copy(onnx, '/content/drive/MyDrive/road_models/defects.onnx')
json.dump(metrics, open('/content/drive/MyDrive/road_models/defects_metrics.json','w'), indent=1)
shutil.copy('runs/detect/defects/weights/best.pt', '/content/drive/MyDrive/road_models/defects_best.pt')
print('Saved to Drive: road_models/defects.onnx, defects_metrics.json, defects_best.pt')